# Análise Exploratória dos Dados de Segurança do ISP

## Estatísticas e Previsão de Segurança: Roubos e Furtos em coletivo no Rio de Janeiro

    O presente estudo originou-se de um processo exploratório de diferentes bases de dados do Instituto de Segurança Pública (ISP). Inicialmente, foram avaliados dados sobre letalidade violenta e feminicídios. No entanto, ambas as bases mostraram-se insuficientes para uma análise estatística aprofundada: a primeira por apresentar baixa densidade amostral e a segunda pelo volume reduzido de registros para o desenvolvimento de um modelo. Dessa forma, optou-se pela utilização do dataset "Estatísticas de segurança: série histórica mensal por área de delegacia", que oferece dados geolocalizados desde 2003. O foco do estudo foi delimitado para a segurança no transporte público, especificamente o recorte de crimes ocorridos em coletivos na Capital do Estado e Baixada Fluminense.

    Nesse notebook faremos uma análise exploratória inicial do nosso dataset. Temos como objetivo final prever a criminalidade em ônibus em cada região com base na série temporal. Vamos, inicialmente, avaliar o conteúdo do dataset, verificar quais colunas podem ser úteis e fazer uma análise geral delas.

### Configurações iniciais do projeto

Carregando as bibliotecas e definindo a paleta de cores:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from faz_mapa import faz_mapa
from IPython.display import IFrame

In [ ]:
sequentialpalette = sns.color_palette('inferno')
qualitativepalette = sns.color_palette('colorblind')

sns.set_palette(palette=sequentialpalette)

sns.set_style("whitegrid")

Carregando o dataset para análise:
Limpeza dessa base no arquivo fix.ipynb.

In [ ]:
with open("../datasets/dataframe.pkl","rb") as file:
    df = pickle.load(file)
    
df.info()

O dataset possui muitas colunas. A maioria parece não muito relacionada ao nosso problema.
Também, precisamos verificar se há dados nulos:

In [ ]:
df.isna().sum()

Verificamos o histórico da base:

print(f"Min: {df['mes_ano'].min()}, Max: {df['mes_ano'].max()}")

Temos dados desde Janeiro de 2014 até Março de 2026.
Quantidade de resgistros no dataset:

In [ ]:
print("Número de registros:", len(df))

As colunas"roubo_em_coletivo" e "furto_coletivo" dizem respeito a segurança no transporte público, serão usadas por nós. Além disso, temos os campos "cisp", "aisp" e "risp", que servem de proxy com diferentes níveis de granularidade para as regiões para nas quais os incidentes foram reportados. Podemos ver mais a respeito sobre essas regiões [nesse documento](https://www.ispdados.rj.gov.br/Arquivos/Relacaodas%20RISP_AISP.pdf).

### Exploração de furtos e roubos em coletivos

Vamos analisar incialmente as principais variáveis de interesse do estudo: roubo_em_coletivo e furto_coletivo por CISP, RISP ou AISP. Sendo CISP o georreferenciamento mais granular disponível na base de dados.

In [ ]:
df["crime_coletivo"] = df["roubo_em_coletivo"] + df["furto_coletivo"]
df_coletivo = df[["crime_coletivo","furto_coletivo","roubo_em_coletivo", "cisp"]]

In [ ]:
print(df_coletivo.groupby("cisp"))

In [ ]:
df_map = df_coletivo.groupby("cisp")["crime_coletivo"].sum().reset_index()
faz_mapa("Crime contra Õnibus Total", "Número de crimes", df_map,["cisp","crime_coletivo"],"cisp")
IFrame(src="./Crime contra Õnibus Total.html", width=700, height=400)

Vemos que a região com mais incidência é a de bonsucesso. Essa região tem o entroncamento entre duas principais vias da cidade (Avenida Brasil e Linha amarela), além de ser cortada pela Linha Vermelha. Ela também abrica o complexo da maré, que fica as margens dessas vias. Nesse sentido, é bem esperado essa ser a região campeã. No entando, os segundo e terceiro lugar serem Barra e Jacarepaguá é surpreendente. Porém esses dados são agregados de um longo período de tempo, precisamos explorar melhor.
Nesse sentido, queremos entender a relação entre os crimes de furto e roubo em coletivo. Para isso, podemos calcular a correlação entre essas duas variáveis. A correlação é uma medida estatística que indica o grau de associação entre duas variáveis.

In [ ]:
df[["furto_coletivo", "roubo_em_coletivo"]].corr()

Correlação entre furto_coletivo e roubo_em_coletivo é 0.198912, na análise estatística, essa correlação é considerada desprezível ou muito baixa. Essa correlação muito baixa indica que o aumento de uma modalidade não implica necessariamente no crescimento da outra no estado.

Agora vamos analisar a evolução dos crimes ao longo dos anos. Vamos criar gráficos de barras para cada tipo de crime, mostrando a soma total por ano.

In [ ]:
plt.figure(figsize=(8,3))
plt.title("Total de roubos por ano")
ax = sns.barplot(data=df, x="ano", y="total_roubos", estimator="sum", palette=qualitativepalette, errorbar=None)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')
plt.figure(figsize=(8,3))
plt.title("Total de roubos em coletivos por ano")
ax = sns.barplot(data=df, x="ano", y="roubo_em_coletivo", estimator="sum", palette=qualitativepalette, errorbar=None)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')

plt.figure(figsize=(8,3))
plt.title("Total de furtos por ano")
ax = sns.barplot(data=df, x="ano", y="total_furtos", estimator="sum", palette=qualitativepalette, errorbar=None)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')
plt.figure(figsize=(8,3))
plt.title("Total de furtos em coletivos por ano")
ax = sns.barplot(data=df, x="ano", y="furto_coletivo", estimator="sum", palette=qualitativepalette, errorbar=None)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f')

Vemos que os furtos e roubos em coletivo são uma parte bem pequena dos furtos e roubos totais.

### Buscando correlações entre as colunas

Primeiramente, uma visão geral da correlação:

In [ ]:
dfnumerical = df.select_dtypes(include=np.number)

plt.figure(figsize=(24,20))
sns.heatmap(dfnumerical.corr())

Se tratam de muitas colunas, fica dificil tirar qualquer conclusão. Selecionando mais delicadamente quais colunas visualizar no heatmap:

In [ ]:
# Fazendo o heatmap somente dos valores que ficam acima de um dado valor (threshold)
threshold = 0.5

dfheatmapcorrfurto = dfnumerical[dfnumerical.corr()[dfnumerical.corr()["furto_coletivo"] > threshold]["furto_coletivo"].keys()].corr()
plt.figure(figsize=(12,10))
sns.heatmap(dfheatmapcorrfurto, annot=True, annot_kws={"size": "x-small"}).set_title(f"Correlação com Furto em Coletivo > {threshold}")

In [ ]:
dfheatmapcorrroubo = dfnumerical[dfnumerical.corr()[dfnumerical.corr()["roubo_em_coletivo"] > threshold]["roubo_em_coletivo"].keys()].corr()
plt.figure(figsize=(12,10))
sns.heatmap(dfheatmapcorrroubo, annot=True, annot_kws={"size": "xx-small"}).set_title(f"Correlação com Roubo em Coletivo > {threshold}")

Nesses heatmaps, vemos correlações apenas com crimes da mesma "estirpe": roubo com roubo, furto com furto, a maioria já esperado. O maior insight aqui é que roubos e furtos em coletivos tem baixa correlação entre si, como já prevemos. Para avaliar melhor, vamos plotar novamente o mapa de crimes, dessa vez separadamente:

In [ ]:
df_map = df_coletivo.groupby("cisp")["furto_coletivo"].sum().reset_index()
faz_mapa("Furto contra Õnibus Total", "Número de crimes", df_map,["cisp","furto_coletivo"],"cisp")
IFrame(src="./Furto contra Õnibus Total.html", width=700, height=400)

In [ ]:
df_map = df_coletivo.groupby("cisp")["roubo_em_coletivo"].sum().reset_index()
faz_mapa("Roubo contra Õnibus Total", "Número de crimes", df_map,["cisp","roubo_em_coletivo"],"cisp")
IFrame(src="./Roubo contra Õnibus Total.html", width=700, height=400)

A distinção entre as modalidades criminais torna-se evidente ao cruzar a natureza do delito com sua distribuição geográfica. Observa-se que os roubos em coletivos (caracterizados pelo uso de violência ou grave ameaça) apresentam maior prevalência em regiões da Zona Norte, Zona Oeste e Jacarepaguá. Em contrapartida, os furtos (subtração sem ameaça direta) concentram-se em áreas como a Zona Sul e Barra da Tijuca.
Essa segregação espacial explica a baixa correlação (0,1989) encontrada anteriormente: os crimes respondem a contextos socioeconômicos e oportunidades ambientais distintas. Enquanto o roubo pode estar vinculado a rotas de maior vulnerabilidade e ausência de policiamento ostensivo, o furto parece seguir a lógica da oportunidade em locais de alta densidade de bens de valor e fluxo de transeuntes. Essa heterogeneidade justifica a decisão de modelar cada variável de forma independente, utilizando tratamentos específicos para cada localidade.

### Roubos e furtos em coletivos através do tempo

Vamos valiar a evolução dos roubos e furtos totais ao longo dos anos. Iremos caracterizar por RISPs, que são um nível menos granular que as CISPs, para facilitar a visualização inicialmente.

Primeiro, vamos ver um histograma mostrando a quantidade total de roubos e furtos em todos os anos por RISP.

In [ ]:
plt.figure(figsize=(16,2))
sns.barplot(data=df, x="ano", y="roubo_em_coletivo", hue="risp", estimator="sum", palette=qualitativepalette)
plt.figure(figsize=(16,2))
sns.barplot(data=df, x="ano", y="furto_coletivo", hue="risp", estimator="sum", palette=qualitativepalette)

Observa-se, através da categorização por RISP, que o impacto da pandemia de COVID-19 gerou um comportamento atípico em todas as regiões, com quedas acentuadas nos registros. No entanto, a retomada apresenta distinções cruciais entre as modalidades: os furtos em coletivos recuperaram-se rapidamente, superando a tendência dos roubos, que permanecem em níveis de estabilidade inferiores aos picos de 2016. Essa evolução distinta reforça a independência estatística entre furtos e roubos. 
Tomou-se então, a decisão de realizar o recorte temporal a partir de 2021, fundamentada na mudança estrutural na dinâmica urbana do Rio de Janeiro. O cenário da mobilidade urbana e a dinâmica criminal sofreram alterações drásticas entre 2020 e 2021, tornando os dados pré-pandêmicos menos precisos para prever comportamentos atuais. A pandemia de COVID-19 alterou o fluxo de passageiros no transporte público. Dados anteriores a 2021 refletem um comportamento de deslocamento que não condiz com a realidade atual de regimes híbridos de trabalho e novas rotas comerciais. Ao focar no período pós-2021, garantimos que o modelo capture a "mancha criminal do novo normal". Modelos preditivos funcionam melhor quando treinados com dados que representam o comportamento atual do sistema. Como o objetivo é prever os próximos 5 anos e sugerir rotas seguras entre bairros, utilizar dados de 5 ou 10 anos atrás traria um viés de realidade que não existe mais nas ruas. 

In [ ]:
df = df[df['ano'] >= 2021]

# Novo intervalo de tempo
print(f"Min: {df['mes_ano'].min()}, Max: {df['mes_ano'].max()}")

# Verificando as informações do DataFrame após o filtro
df.info()
print(df.shape)

Em seguida, vamos ver um histograma mostrando a quantidade total de roubos e furtos por RISP.

In [ ]:
plt.figure(figsize=(12,2))
sns.barplot(data=df, x="ano", y="roubo_em_coletivo", hue="risp", estimator="sum", palette=qualitativepalette)
plt.figure(figsize=(10,2))
sns.barplot(data=df, x="ano", y="furto_coletivo", hue="risp", estimator="sum", palette=qualitativepalette)

Enquanto a Zona Sul (RISP 1) brilha nos furtos devido à oportunidade e ao fluxo de turistas, a RISP 3 concentra uma mancha de roubos (com violência/ameaça), muitas vezes associada a áreas de conflito e rotas de fuga em vias expressas.

In [ ]:
#Boxplot aqui

In [ ]:
#CISPS COM MAIOR CASOS, NO ARQUIVO ANALISEEXPLORATORIA TEM UNS TEXTOES COM CONCLUSOES DESSA PARTE, AÍ PEGA DE LÁ, TA PRO FINAL DO NOTEBOOK

In [ ]:
#Proporção de crime totais e em coletivo <- tambem ta no meu notebook

In [ ]:
# Pensei em daí começar aqui a parte de EDA_temporal_cisps